In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

In [2]:
df = pd.read_csv("digital_payments.csv")

print("Dataset loaded successfully.")
print("Shape:", df.shape)

print("\nFirst 5 rows:")
print(df.head())

Dataset loaded successfully.
Shape: (300, 10)

First 5 rows:
   Transaction_ID  Age    Location Payment_Method Merchant_Category  \
0               1   59     Wangdue     QR Payment   Online Shopping   
1               2   18      Mongar     QR Payment   Online Shopping   
2               3   46     Punakha    Credit Card            Retail   
3               4   62     Wangdue  Mobile Wallet            Retail   
4               5   66  Trashigang  Mobile Wallet   Online Shopping   

   Transaction_Amount  Transaction_Hour Device_Type  Customer_Tenure_Months  \
0              928.13                13         iOS                      38   
1              253.50                22         iOS                      28   
2              158.85                21         Web                      25   
3              345.61                19     Android                      10   
4              285.06                16     Android                      60   

  Is_High_Value  
0           Yes  
1

In [3]:
print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

df = df.drop_duplicates()

# Remove rows with missing target
df = df.dropna(subset=["Is_High_Value"])


Missing values:
Transaction_ID            0
Age                       0
Location                  0
Payment_Method            0
Merchant_Category         0
Transaction_Amount        0
Transaction_Hour          0
Device_Type               0
Customer_Tenure_Months    0
Is_High_Value             0
dtype: int64

Duplicate rows: 0


In [4]:
df["Is_High_Value"] = (
    df["Is_High_Value"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({
        "yes": 1,
        "no": 0
    })
)

# Remove rows where target could not be converted
df = df.dropna(subset=["Is_High_Value"])

df["Is_High_Value"] = df["Is_High_Value"].astype(int)

In [5]:
X = df.drop(columns=["Is_High_Value"])

y = df["Is_High_Value"]


# Remove transaction ID if available
if "Transaction_ID" in X.columns:
    X = X.drop(columns=["Transaction_ID"])


In [6]:
categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

print("\nCategorical features:")
print(categorical_features)

print("\nNumerical features:")
print(numerical_features)


Categorical features:
['Location', 'Payment_Method', 'Merchant_Category', 'Device_Type']

Numerical features:
['Age', 'Transaction_Amount', 'Transaction_Hour', 'Customer_Tenure_Months']


C:\Users\FUJITSU\AppData\Local\Temp\ipykernel_10224\3310229653.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(


In [7]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            StandardScaler(),
            numerical_features
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ]
)

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [9]:
models = {

    "Logistic Regression": LogisticRegression(
        max_iter=1000
    ),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=5,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )
}

In [ ]:
results = []

trained_models = {}

for name, model in models.items():

    print("\nTraining:", name)

    pipeline = Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor
            ),
            (
                "model",
                model
            )
        ]
    )

    pipeline.fit(
        X_train,
        y_train
    )

    y_pred = pipeline.predict(X_test)

    y_probability = pipeline.predict_proba(
        X_test
    )[:, 1]

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    auc = roc_auc_score(
        y_test,
        y_probability
    )

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC AUC": auc
    })

    trained_models[name] = pipeline

    print("Accuracy :", round(accuracy, 4))
    print("Precision:", round(precision, 4))
    print("Recall   :", round(recall, 4))
    print("F1 Score :", round(f1, 4))
    print("ROC AUC  :", round(auc, 4))

IndentationError: unexpected indent (3501560584.py, line 70)